# 21. The neural model on the target-encoded features

**One variable against ledger row 16** (`neural_mlp_kaggle`, CV 0.939169): the same
architecture, the same hyperparameters, the same folds, the same seed, the same
early-stopping rule. Only the feature set changes.

## Why this is being run

Row 16 trained on the **raw** 12 features and landed 0.0247 behind the LightGBM
blend. Target encoding did not exist yet. Nothing has ever trained a neural model on
the encoded features, and the argument that reopened CatBoost on 2026-08-17 applies
here without modification: a model rejected under one representation was never
re-tested under the one that turned out to matter.

The CatBoost precedent is the reason to expect something. On the raw features CatBoost
was 0.001675 behind LightGBM; on the encoded ones it is 0.000157 ahead, a swing of
about 0.0018. Whether that transfers here is unknown, and the mechanism is different:
CatBoost gained because its ordered target statistics went from touching 3 of 12
columns to being redundant with all of them, which is not a story about neural nets.

**What makes this worth a GPU session anyway** is that the neural model is the only
genuinely different mechanism in the stack. It sits 0.0247 behind and still earns
**+0.118** in the fitted combiner, because it is wrong in a direction nothing else is.
If the representation lifts it anywhere near the GBDTs while it stays differently
wrong, it is worth more than anything since row 17.

**And it may not work at all.** The encoded columns are smoothed target means, which
is a representation trees split cheaply and an MLP may gain nothing from, since an MLP
was never paying the many-splits cost that made target encoding worth +0.0033 for
LightGBM. A null result here is a real answer and gets a ledger row either way.

## What is held fixed

Everything from row 16: 8-dimensional embeddings for the three categoricals with a
reserved slot for missing, quantile transform to normal, missing-mask columns,
512/256/128 with BatchNorm, SiLU and 0.2 dropout, AdamW under OneCycle, 30 epochs,
early stopping on fold AUC with patience 5, batch 4096.

The numeric block grows from 9 columns to 33, the 9 originals plus the 24 target and
frequency encodings. The three categoricals keep their embeddings **and** gain their
encodings, which is duplicative in exactly the way row 17 was duplicative for
LightGBM, and is what makes this one variable rather than two.

## Leakage

The encoder is fit inside the fold loop with the inner KFold-5 nesting from `13`, and
the quantile transform and the median imputation are fit on training rows only, inside
the same loop. Both are checked by execution below before anything trains.


In [ ]:
# SMOKE trims everything to a couple of minutes on CPU. Run it once before spending a
# GPU session: it exercises every line on a small subset, so a crash costs two minutes
# instead of a wasted session.
SMOKE = True

SEED = 42
N_SPLITS = 5
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# Every one of these is row 16's value and none of them is touched.
EPOCHS, BATCH, LR, WD = 30, 4096, 3e-3, 1e-4
PATIENCE = 5
EMB_DIM, HIDDEN, DROPOUT = 8, (512, 256, 128), 0.20
USE_MISSING_MASK = True

# Ledger row 16, the same architecture on the raw 12 features. This is the number the
# experiment is measured against, paired per fold.
ROW16_CV = 0.939169
# Row 17 and row 26, for scale only. A neural model is not expected to reach these.
ROW17_CV = 0.966782
ROW26_CV = 0.966915

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

if SMOKE:
    EPOCHS, N_SPLITS = 2, 2

print(f"SMOKE = {SMOKE}")


In [ ]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")


## The encoder, and the check that it is the same one

Copied from `13_target_encoding.ipynb`, like `17` and `19`. The copy is fingerprinted
through `ast.unparse`, which strips comments and formatting but keeps semantics, so a
drifted copy is caught rather than assumed away. Under the notebook layout this is the
substitute for a config hash. It has caught a real drift once already.


In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")


In [ ]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()


In [ ]:
# Categorical codes. Fit on train and test together, which is safe: it uses no target
# information whatsoever, only the set of levels that exist. Unchanged from row 16.
cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}      # 0 reserved for missing
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)

Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

# The mask is taken on the ORIGINAL numeric columns only. The encoded columns have no
# NaN by construction, since _apply sends an unseen level to the prior.
mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)


class TabMLP(nn.Module):
    def __init__(self, n_num, cat_sizes, emb_dim=EMB_DIM, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim) for s in cat_sizes])
        dim = n_num + emb_dim * len(cat_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat([xn] + e, dim=1)).squeeze(1)


# num_workers > 0 deadlocks under Jupyter on Windows, which is where SMOKE runs.
NUM_WORKERS = 0 if os.name == "nt" else 2


def make_loader(xn, xc, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(
        ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEV.type == "cuda"), drop_last=drop_last)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xn, xc, _ in loader:
        out.append(torch.sigmoid(model(xn.to(DEV), xc.to(DEV))).float().cpu().numpy())
    return np.concatenate(out)


LOG = OUT / "21_neural_te.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, device={DEV}, folds={N_SPLITS} ===")


In [ ]:
oof = np.zeros(len(train), dtype=np.float64)
test_pred = np.zeros(len(test), dtype=np.float64)
fold_scores = []
t_start = time.time()

for f in range(N_SPLITS):
    seed_all(SEED + f)
    tr_i = np.where(folds != f)[0]
    va_i = np.where(folds == f)[0]

    # The encoder runs inside the fold. Training rows get inner out-of-fold values,
    # validation and test rows get a fit that never saw the validation fold.
    Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
    num_tr = Etr[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)
    num_va = Eva[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)
    num_te = Ete[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)

    # Imputation and the quantile transform fit on training rows only.
    med = np.nanmedian(num_tr, axis=0)

    def prep(a):
        return np.where(np.isnan(a), med, a)

    qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                             subsample=200_000, random_state=SEED)
    qt.fit(prep(num_tr))

    def finish(a, m):
        x = qt.transform(prep(a)).astype(np.float32)
        return np.hstack([x, m]) if USE_MISSING_MASK else x

    Xn_tr = finish(num_tr, mask_tr[tr_i])
    Xn_va = finish(num_va, mask_tr[va_i])
    Xn_te = finish(num_te, mask_te)

    tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], y[tr_i], BATCH, True, drop_last=True)
    va_loader = make_loader(Xn_va, Xc_tr[va_i], None, BATCH * 4, False)
    te_loader = make_loader(Xn_te, Xc_te, None, BATCH * 4, False)

    model = TabMLP(Xn_tr.shape[1], cat_sizes).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr_loader))
    lossf = nn.BCEWithLogitsLoss()

    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        for xn, xc, yy in tr_loader:
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(xn.to(DEV), xc.to(DEV)), yy.to(DEV))
            loss.backward()
            opt.step()
            sched.step()
        auc = roc_auc_score(y[va_i], predict(model, va_loader))
        if auc > best_auc:
            best_auc, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep % 5 == 0 or bad >= PATIENCE:
            print(f"  fold {f} epoch {ep:>2}: val AUC {auc:.6f} (best {best_auc:.6f})")
        if bad >= PATIENCE:
            print(f"  early stop at epoch {ep}")
            break

    model.load_state_dict(best_state)
    oof[va_i] = predict(model, va_loader)
    test_pred += predict(model, te_loader) / N_SPLITS
    fold_scores.append(float(roc_auc_score(y[va_i], oof[va_i])))
    el = time.time() - t_start
    note(f"fold {f}: AUC {fold_scores[-1]:.6f}   elapsed {el/60:.1f} min, "
         f"about {el/(f+1)*(N_SPLITS-f-1)/60:.1f} min left")

cv_mean, cv_std = float(np.mean(fold_scores)), float(np.std(fold_scores))
print()
print(f"neural TE CV {cv_mean:.6f} +/- {cv_std:.6f} "
      f"in {(time.time()-t_start)/60:.1f} min")


In [ ]:
# Paired against ledger row 16 on the identical folds. The right test when two models
# share folds is the spread of the per-fold DIFFERENCES and how many folds it wins,
# not the fold spread, which is common to both and cancels. See NOTES.md.
print(f"row 16, same architecture on the raw 12 features : {ROW16_CV:.6f}")
print(f"this run, on the 36 encoded features             : {cv_mean:.6f}")
print(f"difference                                       : {cv_mean - ROW16_CV:+.6f}")

try:
    base = np.load(locate("neural_oof.npy"))[ROW_IDX]
    bf = np.array([roc_auc_score(y[folds == f], base[folds == f])
                   for f in range(N_SPLITS)])
    print(f"\nrow 16's own vector reproduces its ledger CV to "
          f"{bf.mean() - ROW16_CV:+.2e}")
    d = np.array(fold_scores) - bf
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"paired: {d.mean():+.6f}, sd {d.std(ddof=1):.6f}, "
          f"{(d > 0).sum()}/{N_SPLITS} folds, t={t:.2f}")
    print("  per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    # NOTES.md: a jump over ~2% is treated as a leak until proven otherwise.
    print(f"relative change {(cv_mean - ROW16_CV) / ROW16_CV:+.4%} "
          f"(>2% means stop and investigate before believing it)")
except FileNotFoundError:
    print("\nrow 16's out-of-fold vector not attached, so the paired test is skipped")

print()
print(f"for scale only, not a target: row 17 LightGBM {ROW17_CV:.6f}, "
      f"row 26 CatBoost {ROW26_CV:.6f}")
print("The decision is NOT this CV. It is what a FITTED combiner does with the")
print("vector, measured in the follow-up stack notebook where all 23 members live.")
print("Row 16 scored 0.939169, lost at every equal weight, and still earns +0.118")
print("in the fitted stack, which is the whole reason this run exists.")


In [ ]:
pre = "SMOKE_" if SMOKE else ""
np.save(OUT / f"{pre}neural_te_oof.npy", oof)
np.save(OUT / f"{pre}neural_te_test.npy", test_pred)
pd.DataFrame({ID: test[ID], TARGET: test_pred}).to_csv(
    OUT / f"{pre}neural_te_submission.csv", index=False)
print(f"wrote {pre}neural_te_oof.npy, {pre}neural_te_test.npy, "
      f"{pre}neural_te_submission.csv")
print()
print("ledger line:")
print(f"  name    neural_te")
print(f"  cv_mean {cv_mean:.6f}")
print(f"  cv_std  {cv_std:.6f}")
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}")
